# German Replication of Greenwood, Shleifer & You (2019)

This notebook presents the executed German replication/adaptation of *Bubbles for Fama*. The underlying Stata workflow is the authoritative computation engine. This notebook reads the executed project files and presents the results in a clean seminar-paper format.


## 1. Project Setup

The project is a replication/adaptation of Greenwood, Shleifer, and You (2019). The notebook is a presentation layer rather than a raw-data calculation engine. The analysis is restricted to Germany and relies on the executed Stata output files.


## 2. Portable Project-File Discovery

The project directory is discovered using `pathlib` from the current working directory and nearby candidate folders. This keeps the notebook portable across machines.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

EXPECTED_FILES = [
    'germany_crash_results.xlsx',
    'germany_crash_test.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_event_time_paths.xlsx',
    'germany_table1_paper_style.xlsx',
    'germany_table3_main.xlsx',
    'germany_replication_checks.xlsx',
    'germany_event_time_paths.png',
]

def find_project_dir():
    cwd = Path.cwd()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.append(base)
        try:
            candidates.extend(p for p in base.iterdir() if p.is_dir())
        except OSError:
            pass
    for base in [cwd, *cwd.parents]:
        for child in ['data', 'project', 'results', 'files', 'input', 'inputs']:
            candidate = base / child
            if candidate.is_dir():
                candidates.append(candidate)
    scored = []
    for candidate in dict.fromkeys(candidates):
        if not candidate.exists() or not candidate.is_dir():
            continue
        present = sum((candidate / name).exists() for name in EXPECTED_FILES)
        score = present * 100
        if (candidate / 'germany_crash_results.xlsx').exists():
            score += 10
        if (candidate / 'germany_event_time_paths.png').exists():
            score += 10
        scored.append((score, candidate, present))
    if not scored or max(item[0] for item in scored) <= 0:
        raise FileNotFoundError(
            'No project directory matched the expected Germany replication result files.'
        )
    scored.sort(key=lambda x: (-x[0], -x[2], str(x[1])))
    return scored[0][1]

PROJECT_DIR = find_project_dir()
display(Markdown(f'**Project directory:** `{PROJECT_DIR}`'))
for name in EXPECTED_FILES:
    print(f'{name}: {"FOUND" if (PROJECT_DIR / name).exists() else "MISSING"}')

def read_result_excel(filename):
    path = PROJECT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Required executed result file not found: {filename}')
    return pd.read_excel(path)


## 3. Research Design and Data Constraint

The German replication follows the original paper’s framework: large run-ups, subsequent crash and drawdown behavior, raw and net-of-market returns, volatility controls, event-time analysis, and Footnote-15-style regressions. The empirical project is restricted to a single WRDS extract and uses only the supplied German data.


## 4. Source Data and Sample Construction

The analytical file is `comp_global_daily.dta`. The country filter is `fic == "DEU"`. The unit of analysis is the industry, not a single German index. Security identifiers are `gvkey + iid`, common equity is `tpci == "0"`, and the primary GICS level is `gsector`.


## 5. Daily Security Cleaning and Currency Normalization

The adjusted total-return price is constructed as `(prccd * fx_eur / ajexdi) * trfd`, and the reporting currency is EUR. DEM observations are converted at the fixed rate `1 EUR = 1.95583 DEM`. Daily returns are used for volatility calculations, while the main run-up and crash analysis uses month-end data.


## 6. Daily Return Cleaning

The Stata workflow flags observations in which the current return or prior return exceeds 100 percent and the two-day compounded return is below 20 percent. Any current return above 200 percent is also flagged and set to missing. This is part of the empirical design and is not recomputed here.


## 7. Monthly Returns and Industry Portfolio Construction

Monthly industry portfolios are value-weighted using lagged market capitalization based on `cshoc`. The benchmark market portfolio is constructed from the same German securities.


## 8. Identification of Large Price Increases

For thresholds 50%, 75%, 100%, 125%, and 150%, the strict main definition requires: past 24-month raw return >= X; past 24-month net-of-market return >= X; and past 60-month raw return >= 50%. The signal is lagged one month, only the first qualifying month is retained, and the following 24 months are suppressed.


## 9. Subsequent Crash Outcomes

A crash is defined as a 40 percent drawdown from a previous peak during the next 24 months. The executed output records crash indicators, forward raw returns, drawdowns, and complete 12M and 24M outcomes.


In [ ]:
crash_results = read_result_excel('germany_crash_results.xlsx')
display(crash_results)


## 10. Pre-Run-Up Volatility

Annualized pre-run-up volatility is constructed from monthly industry returns. The executed volatility file is used for lightweight validation and is not reconstructed in Python.


## 11. Analysis Sample and Merge

The executed volatility and outcome data are merged into the final analysis sample. The notebook reads the merged output and documents the main results without redoing the merge.


## 12. Crash Probabilities Following Large Price Increases

The central test asks whether industries with a large price increase subsequently experience elevated crash risk. The executed result file is read directly.


In [ ]:
crash_test = read_result_excel('germany_crash_test.xlsx')
display(crash_test)


## 13. Conditional vs Unconditional Crash Risk

The comparison is between crash risk after a strict 100 percent run-up and unconditional crash risk in the broader German industry panel. The notebook reads the executed test rather than recomputing it.


## 14. Statistical Tests of Crash Risk

The formal test is a comparison of crash probabilities and associated test statistics. Fisher’s exact test is the critical small-sample diagnostic because the strict 100 percent sample is small.


In [ ]:
footnote15 = read_result_excel('germany_footnote15_regressions.xlsx')
display(footnote15)


## 15. Price Dynamics Around Large Run-Ups

The event study is based on the strict 100 percent run-up episodes. Event month 0 is the first identified run-up month, and the window is -24 through +48 months.


In [ ]:
event_time = read_result_excel('germany_event_time_paths.xlsx')
display(event_time)
img_path = PROJECT_DIR / 'germany_event_time_paths.png'
if not img_path.exists():
    raise FileNotFoundError('Required executed result file not found: germany_event_time_paths.png')
display(Image(filename=str(img_path)))


## 16. Threshold Results

The executed threshold table summarizes the GSY-style results for 50%, 75%, 100%, 125%, and 150%. Risk-free dependent fields are N/A because the project is restricted to the single WRDS source.


In [ ]:
table3_main = read_result_excel('germany_table3_main.xlsx')
display(table3_main)


## 17. Table 1 and the Strict Main Definition

Table 1 uses a raw-only 100 percent definition: past 24-month raw return >= 100 percent and past 60-month raw return >= 50 percent, with the same lagging and suppression rule, but no net-market requirement. That differs from the strict main analysis, which also imposes a net-market requirement.


In [ ]:
table1 = read_result_excel('germany_table1_paper_style.xlsx')
display(table1)


## 18. Replication and Integrity Checks

The execution checks confirm the expected threshold counts, the number of strict 100 percent run-ups, the event-time coverage, and the regression count. The checks are read directly from the executed file.


In [ ]:
replication_checks = read_result_excel('germany_replication_checks.xlsx')
display(replication_checks)


## 19. Validating Executed Intermediate Files

The project includes executed intermediate files such as the daily clean and industry-monthly stages. These are validated for presence and coherence but are not recomputed within Python.


In [ ]:
for name in [
    'germany_daily_clean.dta',
    'germany_industry_monthly.dta',
    'germany_runup_monthly.dta',
    'germany_runup_outcomes.dta',
    'germany_industry_volatility.dta',
    'germany_gsy_analysis.dta'
]:
    print(f'{name}: {"FOUND" if (PROJECT_DIR / name).exists() else "MISSING"}')


## 20. Limitations

This project is restricted to one WRDS source, uses no external risk-free series, keeps risk-free fields as N/A, converts DEM using the fixed 1.95583 conversion, relies on a small strict 100 percent sample, and treats the notebook as a reporting layer while Stata remains the raw-data calculation engine.
